In [ ]:
import os
import shutil
import pandas as pd
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
import glob

class B3DataCollector:
    def __init__(self):
        self.DOWNLOAD_PATH = "C://ARQUIVOS//OneDrive//Bolsa//COLETA//B3//WDO//"
        self.CHROMEDRIVER_PATH = "C:\chromedriver\chromedriver.exe"
        self.CACHE_PATH = os.path.expanduser("~/.wdm")
        self.existing_files = self.map_existing_files()
        self.driver = None

    def map_existing_files(self):
        """Mapeia arquivos existentes no diretório"""
        if not os.path.exists(self.DOWNLOAD_PATH):
            os.makedirs(self.DOWNLOAD_PATH)
            return set()

        files = glob.glob(os.path.join(self.DOWNLOAD_PATH, "WDO_*.csv"))
        existing_dates = set()
        
        for file in files:
            # Extrai a data do nome do arquivo (formato: DI1_DDMMYYYY.csv)
            try:
                date_str = os.path.basename(file).split('_')[1].split('.')[0]
                existing_dates.add(date_str)
            except:
                continue
                
        print(f"Encontrados {len(existing_dates)} arquivos existentes")
        return existing_dates

    def limpar_cache_chromedriver(self):
        """Limpa cache do ChromeDriver"""
        if os.path.exists(self.CACHE_PATH):
            shutil.rmtree(self.CACHE_PATH, ignore_errors=True)
            print("Cache do ChromeDriver limpo")

    def configurar_driver(self):
        """Configura e inicia o ChromeDriver"""
        self.limpar_cache_chromedriver()
        options = Options()
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--headless")  # Execução sem interface gráfica
        
        self.driver = webdriver.Chrome(
            service=Service(self.CHROMEDRIVER_PATH), 
            options=options
        )
        return self.driver

    def check_date_exists(self, date_str):
        """Verifica se já existe arquivo para a data"""
        formatted_date = date_str.replace('/', '')
        return formatted_date in self.existing_files

    def download_data_for_date(self, date_str):
        """Download dados para uma data específica"""
        try:
            formatted_date = date_str.replace('/', '')
            if self.check_date_exists(date_str):
                print(f"Arquivo para {date_str} já existe - pulando")
                return False

            print(f"Coletando dados para {date_str}...", end=' ')
            
            url = 'https://www.b3.com.br/pt_br/market-data-e-indices/servicos-de-dados/market-data/historico/derivativos/resumo-estatistico/sistema-pregao/'
            self.driver.get(url)
            
            wait = WebDriverWait(self.driver, 5)
            
            # Navega para o iframe
            iframe = wait.until(EC.presence_of_element_located((By.ID, "bvmf_iframe")))
            self.driver.switch_to.frame(iframe)

            # Preenche data
            campo_data = wait.until(EC.presence_of_element_located((By.ID, "dData1")))
            campo_data.clear()
            campo_data.send_keys(date_str)
            
            # Seleciona DI1
            select_mercadoria = Select(wait.until(EC.presence_of_element_located((By.ID, "comboMerc1"))))
            select_mercadoria.select_by_visible_text("WDO : Dólar Mini - WDO")

            # Coleta tabelas
            tabelas = []
            for i in range(1, 4):
                xpath = f"/html/body/div[1]/div[3]/div/table[1]/tbody/tr/td[{i}]/table"
                try:
                    elemento = wait.until(EC.presence_of_element_located((By.XPATH, xpath)))
                    html = elemento.get_attribute('outerHTML')
                    df = pd.read_html(html)[0]
                    df.fillna('', inplace=True)
                    tabelas.append(df)
                except:
                    continue

            if not tabelas:
                print("Sem dados")
                return False

            # Combina tabelas
            max_rows = max(len(df) for df in tabelas)
            tabelas = [df.reindex(range(max_rows)).fillna('') for df in tabelas]
            combined_df = pd.concat(tabelas, axis=1)

            # Salva arquivo
            output_file = os.path.join(self.DOWNLOAD_PATH, f"WDO_{formatted_date}.csv")
            combined_df.to_csv(output_file, index=False, encoding='utf-8')
            
            self.existing_files.add(formatted_date)
            print("OK")
            return True

        except Exception as e:
            print(f"Erro: {str(e)}")
            return False

    def collect_missing_data(self, days_back=1825):
        """Coleta dados faltantes dos últimos X dias"""
        try:
            self.configurar_driver()
            
            end_date = datetime.now()
            start_date = end_date - timedelta(days=days_back)
            current_date = end_date
            
            print(f"\nVerificando dados de {start_date.strftime('%d/%m/%Y')} até {end_date.strftime('%d/%m/%Y')}")
            
            files_collected = 0
            while current_date >= start_date:
                if current_date.weekday() < 5:  # Apenas dias úteis
                    date_str = current_date.strftime("%d/%m/%Y")
                    if self.download_data_for_date(date_str):
                        files_collected += 1
                current_date -= timedelta(days=1)

            print(f"\nColeta finalizada:")
            print(f"- {files_collected} novos arquivos coletados")
            print(f"- {len(self.existing_files)} arquivos totais")

        except Exception as e:
            print(f"Erro durante a coleta: {str(e)}")
        
        finally:
            if self.driver:
                self.driver.quit()

if __name__ == "__main__":
    collector = B3DataCollector()
    collector.collect_missing_data(days_back=1825)

Encontrados 1254 arquivos existentes

Verificando dados de 21/11/2019 até 19/11/2024
Coletando dados para 19/11/2024... Sem dados
Coletando dados para 18/11/2024... 

C:\Users\xthia\AppData\Local\Temp\ipykernel_26328\3333881683.py:102: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html)[0]
C:\Users\xthia\AppData\Local\Temp\ipykernel_26328\3333881683.py:102: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html)[0]
C:\Users\xthia\AppData\Local\Temp\ipykernel_26328\3333881683.py:102: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html)[0]


OK
Coletando dados para 15/11/2024... Sem dados
Arquivo para 14/11/2024 já existe - pulando
Arquivo para 13/11/2024 já existe - pulando
Arquivo para 12/11/2024 já existe - pulando
Arquivo para 11/11/2024 já existe - pulando
Arquivo para 08/11/2024 já existe - pulando
Arquivo para 07/11/2024 já existe - pulando
Arquivo para 06/11/2024 já existe - pulando
Arquivo para 05/11/2024 já existe - pulando
Arquivo para 04/11/2024 já existe - pulando
Arquivo para 01/11/2024 já existe - pulando
Arquivo para 31/10/2024 já existe - pulando
Arquivo para 30/10/2024 já existe - pulando
Arquivo para 29/10/2024 já existe - pulando
Arquivo para 28/10/2024 já existe - pulando
Arquivo para 25/10/2024 já existe - pulando
Arquivo para 24/10/2024 já existe - pulando
Arquivo para 23/10/2024 já existe - pulando
Arquivo para 22/10/2024 já existe - pulando
Arquivo para 21/10/2024 já existe - pulando
Arquivo para 18/10/2024 já existe - pulando
Arquivo para 17/10/2024 já existe - pulando
Arquivo para 16/10/2024 já e